# 第1部 IR データを集める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asg1223/rcj-soccer-ml/blob/main/notebooks/01_ir_data.ipynb)

このノートブックは、教材のページのコードを順番に並べたものです（`tools/build_notebooks.py` で自動生成）。
上のセルから順に `Shift + Enter` で実行してください。

In [ ]:
# Colab のときだけ、教材リポジトリを取ってきて読み込めるようにする
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q --depth 1 https://github.com/Asg1223/rcj-soccer-ml.git /content/rcj-soccer-ml
    sys.path.insert(0, "/content/rcj-soccer-ml")
print("Colab で実行中" if IN_COLAB else "ローカルで実行中")

## 1-1 IR ボールとセンサの仕組み

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part1-ir-data/ball-and-sensor/

### 手を動かす：くせを1つずつ足してみる

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
from rcjml import load_config
from rcjml.sim import simulate, DIFFICULTIES
from rcjml.plot import polar

cfg = load_config()
easy = DIFFICULTIES["easy"]

values = simulate(cfg, 45, 40, difficulty=easy, rng=np.random.default_rng(0))[0]
ax = plt.subplot(projection="polar")
polar(ax, cfg, values, true_angle=45, title="easy")
plt.show()

In [ ]:
from rcjml.plot import polar_grid

variants = {
    "easy": easy,
    "gain / offset": replace(easy, gain_spread=0.3, offset_spread_deg=8.0),
    "body shadow": replace(easy, body_shadow=0.7),
    "ambient light": replace(easy, ambient_mean=1.0, ambient_dir=4.0),
    "reflection": replace(easy, reflection_prob=1.0, reflection_coef=(0.5, 0.7)),
    "occlusion": replace(easy, occlusion_prob=1.0, occlusion_keep=(0.1, 0.1)),
    "stray IR": replace(easy, stray_prob=1.0, stray_level=(30.0, 60.0)),
    "threshold / levels": replace(easy, threshold=1.5, levels=32, jitter=0.18),
}

rows = [simulate(cfg, 45, 40, difficulty=d, rng=np.random.default_rng(1))[0] for d in variants.values()]
polar_grid(cfg, rows, titles=list(variants), true_angles=[45] * len(rows))
plt.show()

In [ ]:
from rcjml.sim import sample_session

rng = np.random.default_rng(2)
rows, titles = [], []
for diff in ["normal", "hard"]:
    session = sample_session(diff, rng)
    for k in range(4):
        rows.append(simulate(cfg, 45, 40, difficulty=diff, session=session, rng=rng)[0])
        titles.append(f"{diff} #{k + 1}")
polar_grid(cfg, rows, titles=titles, true_angles=[45] * len(rows))
plt.show()

## 1-3 PC に送る

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part1-ir-data/send-to-pc/

### 手を動かす 1：1行を数字にする関数を書く

In [ ]:
def parse_line(line, n):
    """'IR,時刻,s0,s1,...' の行をセンサ値のリストにする。形が違えば None。"""
    line = line.strip()
    if not line.startswith("IR,"):
        return None
    parts = line.split(",")
    if len(parts) != n + 2:          # "IR" と時刻の 2 つ + センサ n 個
        return None
    try:
        return [int(p) for p in parts[2:]]
    except ValueError:
        return None

tests = [
    "IR,1538,412,388,96,0,0,0,0,0,0,0,51,302",   # 正しい行
    "# RCJML IR N=12 MAX=1023 WINDOW_US=2500",   # お知らせの行
    "IR,1540,412,388,96,0,0",                    # 途中で切れた行
    "IR,1541,41?,388,96,0,0,0,0,0,0,0,51,302",   # 文字化けした行
]
for t in tests:
    print(parse_line(t, 12))

## 1-4 正解ラベルをつける

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part1-ir-data/labels/

### 手を動かす 2：集める順番を決める

In [ ]:
from rcjml.logger import make_plan

plan = make_plan(seed=1)
print("か所の数:", len(plan))
for spot in plan[:8]:
    print(spot)

samples = 20                 # 1か所で記録する回数
seconds_per_spot = 15        # ボールを置きかえる時間の目安
print(f"1回の取り直し：{len(plan) * samples} 行、だいたい {len(plan) * seconds_per_spot / 60:.0f} 分")

## 1-5 データを眺める

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part1-ir-data/explore/

#### 1. 読み込む

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rcjml import load_config
from rcjml.data import load_synthetic, sensor_columns

cfg = load_config()
df = load_synthetic("ring12_large_normal")
S = sensor_columns(df)          # ['s0', 's1', ..., 's11']

print(df.shape)                 # (行の数, 列の数)
df.head()

#### 2. 数と範囲を確かめる

In [ ]:
print(df.groupby("session").size())          # 取り直しごとの行数
print(df[S].describe().round(0))              # センサごとの平均・最小・最大など
print("0 の割合     :", (df[S] == 0).mean().mean().round(2))
print("最大値の割合 :", (df[S] == cfg.ir.max_reading).mean().mean().round(3))

#### 3. いくつか描いてみる

In [ ]:
from rcjml.plot import polar_grid

sample = df.sample(8, random_state=0)
titles = [f"{a:+.0f} deg, {d:.0f} cm" for a, d in zip(sample.angle_deg, sample.distance_cm)]
polar_grid(cfg, sample[S].values, titles=titles, true_angles=sample.angle_deg.values)
plt.show()

#### 4. 反応の地図（ヒートマップ）

In [ ]:
d = df[df.has_ball == 0].copy()
d["angle_bin"] = (d["angle_deg"] // 15) * 15 + 7.5     # 15° ごとの区間の真ん中
table = d.groupby("angle_bin")[S].mean()

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(table.T.values, aspect="auto", origin="lower",
               extent=[-180, 180, -0.5, len(S) - 0.5], cmap="viridis")
ax.set_xlabel("ball angle [deg]")
ax.set_ylabel("sensor")
ax.set_yticks(range(len(S)), S)
fig.colorbar(im, label="mean reading")
plt.show()

#### 5. 距離で値がどう変わるか

In [ ]:
front = df[(df.angle_deg.abs() < 15) & (df.has_ball == 0)]

fig, ax = plt.subplots(figsize=(6, 4))
for sid, g in front.groupby("session"):
    ax.scatter(g.distance_cm, g.s0, s=8, alpha=0.6, label=f"session {sid}")
ax.set_xscale("log")
ax.set_xlabel("distance [cm] (log scale)")
ax.set_ylabel("s0")
ax.legend(fontsize=7, ncol=2)
plt.show()

#### 6. 取り直しで条件が変わっているか

In [ ]:
mid = df[df.distance_cm.between(30, 60) & (df.has_ball == 0)]
peak = mid[S].max(axis=1)
print(peak.groupby(mid["session"]).mean().round(0))

#### 7. 従来の方法が大きく外すデータを探す

In [ ]:
from rcjml.sim import baseline_angle, angle_error_deg

df["base_err"] = angle_error_deg(baseline_angle(cfg, df[S].values), df["angle_deg"])
odd = df[df.base_err > 45]
print(f"大きく外したデータ: {len(odd)} / {len(df)} 行")
print("その距離:", odd.distance_cm.describe().round(0).to_dict())

# 合成データだけにある「答え合わせ用の印」で、原因を確かめる
flags = ["reflection", "occluded", "stray"]
print(pd.DataFrame({"全体": df[flags].mean(), "大きく外したデータ": odd[flags].mean()}).round(2))

In [ ]:
worst = odd.sort_values("base_err", ascending=False).head(8)
titles = [f"err {e:.0f} deg" for e in worst.base_err]
polar_grid(cfg, worst[S].values, titles=titles, true_angles=worst.angle_deg.values,
           pred_angles=baseline_angle(cfg, worst[S].values))
plt.show()